# Gold Layer LOAD — Company Risk Intelligence Platform
#### Aggregate Silver -> normalize 0-100 -> weight -> load dims + fact_company_risk

In [0]:
# =========================================================
# CELL 1 — Imports & config
# =========================================================
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

spark = SparkSession.builder.getOrCreate()
CATALOG = "company_risk_intelligence_platform"

def scd_merge(df, target_table, keys):
    if not spark.catalog.tableExists(target_table):
        df.write.format("delta").mode("overwrite").option("mergeSchema","true").saveAsTable(target_table)
        print(f"created {target_table}")
    else:
        tgt = DeltaTable.forName(spark, target_table)
        cond = " AND ".join([f"t.{k}=s.{k}" for k in keys])
        (tgt.alias("t").merge(df.alias("s"), cond)
            .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
        print(f"merged {target_table}")

# read silver helpers
def s(tbl):
    return spark.read.table(f"{CATALOG}.silver.{tbl}")


In [0]:
# =========================================================
# CELL 2 — dim_company  (identity + descriptive attrs)
# =========================================================
dimco = (
    s("dim_company")
    .join(s("ch_overview").select("company_number","company_status","company_type","date_of_creation"),
          "company_number", "left")
    .select(
        "company_id","company_name","company_number","ticker","ticker_safe",
        "company_status","company_type",
        F.to_date("date_of_creation").alias("date_of_creation"),
    )
)
scd_merge(dimco, f"{CATALOG}.gold.dim_company", ["company_id"])
display(spark.read.table(f"{CATALOG}.gold.dim_company"))


In [0]:
# =========================================================
# CELL 3 — dim_date  (calendar over the stock date range)
# =========================================================
bounds = s("yf_stock").select(
    F.min("trading_date").alias("mn"), F.max("trading_date").alias("mx")
).collect()[0]

dim_date = (
    spark.sql(f"SELECT explode(sequence(to_date('{bounds['mn']}'), to_date('{bounds['mx']}'), interval 1 day)) AS full_date")
    .withColumn("date_key", F.date_format("full_date","yyyyMMdd").cast("int"))
    .withColumn("day", F.dayofmonth("full_date"))
    .withColumn("month", F.month("full_date"))
    .withColumn("quarter", F.quarter("full_date"))
    .withColumn("year", F.year("full_date"))
    .withColumn("day_of_week", F.date_format("full_date","EEEE"))
    .select("date_key","full_date","day","month","quarter","year","day_of_week")
)
scd_merge(dim_date, f"{CATALOG}.gold.dim_date", ["date_key"])
print("dim_date rows:", spark.read.table(f"{CATALOG}.gold.dim_date").count())

In [0]:
# =========================================================
# CELL 4 — dim_industry  (distinct sector/industry from yf_info)
# =========================================================
dim_industry = (
    s("yf_info").select(
        F.trim(F.col("sector")).alias("sector_name"),
        F.trim(F.col("industry")).alias("industry_name"),
    ).where(F.col("sector").isNotNull()).distinct()
    .withColumn("industry_id", F.row_number().over(Window.orderBy("sector_name","industry_name")))
    .select("industry_id","sector_name","industry_name")
)
scd_merge(dim_industry, f"{CATALOG}.gold.dim_industry", ["industry_id"])
display(spark.read.table(f"{CATALOG}.gold.dim_industry"))

In [0]:
# =========================================================
# CELL 5 — dim_officer  (governance summary per company)
# =========================================================
dim_officer = (
    s("ch_people").groupBy("company_number").agg(
        F.count("*").alias("board_size"),
        F.sum(F.when(F.col("is_active"), 1).otherwise(0)).alias("active_officers"),
        F.sum(F.when(~F.col("is_active"), 1).otherwise(0)).alias("resigned_officers"),
    )
    # map company_number -> company_id via dim_company
    .join(s("dim_company").select("company_id","company_number"), "company_number", "inner")
    .withColumn("director_churn",
                F.when(F.col("board_size") > 0, F.col("resigned_officers")/F.col("board_size")).otherwise(0.0))
    .withColumn("officer_summary_id", F.col("company_id"))
    .select("officer_summary_id","company_id","board_size","active_officers","resigned_officers","director_churn")
)
scd_merge(dim_officer, f"{CATALOG}.gold.dim_officer", ["officer_summary_id"])
display(spark.read.table(f"{CATALOG}.gold.dim_officer"))


In [0]:


# =========================================================
# CELL 6 — FINANCIAL pillar raw metrics (latest period per company)
# yf_financials is long: ticker_safe, statement_type, metric_name, period_end_date, metric_value
# Strategy: rank periods per (company, statement) so latest = rank 1, prior = rank 2.
# =========================================================
fin = s("yf_financials")

# helper: pull one metric at a given period-rank (1 = latest)
def metric_at(statement, name, rank=1):
    w = Window.partitionBy("ticker_safe").orderBy(F.col("period_end_date").desc())
    return (fin.filter((F.col("statement_type")==statement) & (F.col("metric_name")==name))
              .withColumn("rk", F.row_number().over(w))
              .filter(F.col("rk")==rank)
              .select("ticker_safe", F.col("metric_value").alias("v")))

# Income statement metrics
rev_latest = metric_at("income_statement","Total Revenue",1).withColumnRenamed("v","revenue")
rev_prior  = metric_at("income_statement","Total Revenue",2).withColumnRenamed("v","revenue_prior")
net_income = metric_at("income_statement","Net Income",1).withColumnRenamed("v","net_income")

# Balance sheet metrics  (names per yfinance; adjust if your data differs)
total_debt    = metric_at("balance_sheet","Total Debt",1).withColumnRenamed("v","total_debt")
equity        = metric_at("balance_sheet","Stockholders Equity",1).withColumnRenamed("v","equity")
curr_assets   = metric_at("balance_sheet","Current Assets",1).withColumnRenamed("v","curr_assets")
curr_liab     = metric_at("balance_sheet","Current Liabilities",1).withColumnRenamed("v","curr_liab")

fin_base = (
    s("dim_company").select("company_id","ticker_safe")
    .join(rev_latest, "ticker_safe","left").join(rev_prior,"ticker_safe","left")
    .join(net_income,"ticker_safe","left")
    .join(total_debt,"ticker_safe","left").join(equity,"ticker_safe","left")
    .join(curr_assets,"ticker_safe","left").join(curr_liab,"ticker_safe","left")
    .withColumn("debt_to_equity", F.when(F.col("equity")!=0, F.col("total_debt")/F.col("equity")))
    .withColumn("current_ratio",  F.when(F.col("curr_liab")!=0, F.col("curr_assets")/F.col("curr_liab")))
    .withColumn("net_margin",     F.when(F.col("revenue")!=0, F.col("net_income")/F.col("revenue")))
    .withColumn("revenue_growth", F.when(F.col("revenue_prior")!=0,
                                         (F.col("revenue")-F.col("revenue_prior"))/F.col("revenue_prior")))
    .select("company_id","debt_to_equity","current_ratio","net_margin","revenue_growth")
)
display(fin_base)

In [0]:



# =========================================================
# CELL 7 — MARKET pillar raw metrics (from yf_stock + beta from yf_info)
# =========================================================
stock = s("yf_stock").select("ticker","trading_date","close").withColumnRenamed("ticker","ticker_safe")

w_ord = Window.partitionBy("ticker_safe").orderBy("trading_date")
rets = (stock
    .withColumn("prev_close", F.lag("close").over(w_ord))
    .withColumn("ret", (F.col("close")-F.col("prev_close"))/F.col("prev_close"))
)

# annualised volatility = stdev(daily returns) * sqrt(252)
vol = rets.groupBy("ticker_safe").agg((F.stddev("ret")*F.sqrt(F.lit(252))).alias("annual_volatility"))

# max drawdown = min over time of (close / running_peak - 1)
w_peak = Window.partitionBy("ticker_safe").orderBy("trading_date").rowsBetween(Window.unboundedPreceding, 0)
dd = (stock
    .withColumn("peak", F.max("close").over(w_peak))
    .withColumn("drawdown", F.col("close")/F.col("peak") - 1)
    .groupBy("ticker_safe").agg(F.min("drawdown").alias("max_drawdown"))
)

beta = s("yf_info").select(F.col("ticker"), F.col("beta")).withColumnRenamed("ticker","ticker_real")

mkt_base = (
    s("dim_company").select("company_id","ticker","ticker_safe")
    .join(vol,"ticker_safe","left")
    .join(dd,"ticker_safe","left")
    .join(beta, F.col("ticker")==F.col("ticker_real"),"left")
    .select("company_id","annual_volatility","max_drawdown","beta")
)
display(mkt_base)

In [0]:

# =========================================================
# CELL 8 — GOVERNANCE pillar raw metrics (overview + officer churn)
# =========================================================
gov_base = (
    s("dim_company").select("company_id","company_number")
    .join(s("ch_overview").select("company_number","date_of_creation","accounts_overdue"),
          "company_number","left")
    .join(spark.read.table(f"{CATALOG}.gold.dim_officer").select("company_id","director_churn"),
          "company_id","left")
    .withColumn("company_age_years",
                F.datediff(F.current_date(), F.to_date("date_of_creation"))/365.25)
    .withColumn("accounts_overdue",
                F.when(F.col("accounts_overdue")==True,1).otherwise(0))
    .select("company_id","company_age_years","accounts_overdue","director_churn")
)
display(gov_base)



In [0]:
# =========================================================
# CELL 9 — NEWS pillar raw metrics (volume + naive sentiment)
# =========================================================
NEG = ["loss","fall","drop","cut","fraud","probe","lawsuit","decline","warn","slump","plunge","downgrade"]
POS = ["gain","rise","beat","growth","profit","surge","upgrade","record","win","boost","strong"]

# 1. Read news, derive ticker_safe from file_path, lowercase the title
news = (
    s("yf_news")
    .withColumn("ticker_safe",
                F.regexp_extract(F.col("file_path"), r"/\d{4}/\d{2}/\d{2}/([^/]+)/", 1))
    .select("ticker_safe", F.lower(F.coalesce(F.col("title"), F.lit(""))).alias("t"))
)

# 2. Count positive / negative keyword hits in each title
news = (
    news
    .withColumn("neg", sum([F.col("t").contains(w).cast("int") for w in NEG]))
    .withColumn("pos", sum([F.col("t").contains(w).cast("int") for w in POS]))
)

# 3. Aggregate to one row per company
news_base = (
    s("dim_company").select("company_id","ticker_safe")
    .join(
        news.groupBy("ticker_safe").agg(
            F.count("*").alias("news_volume"),
            (F.sum("pos") - F.sum("neg")).alias("net_sent_raw")
        ),
        "ticker_safe", "left"
    )
    .withColumn("news_volume", F.coalesce(F.col("news_volume"), F.lit(0)))
    .withColumn("news_sentiment",
                F.when(F.col("news_volume") > 0,
                       F.col("net_sent_raw") / F.col("news_volume")).otherwise(0.0))
    .select("company_id","news_volume","news_sentiment")
)

display(news_base)

In [0]:



# =========================================================
# CELL 10 — Normalize each metric 0-100 across the universe and score pillars
# higher score = higher risk, so RISK-INCREASING metrics map normal,
# RISK-REDUCING metrics are INVERTED.
# =========================================================
def norm(df, src, out, invert=False):
    stats = df.select(F.min(src).alias("mn"), F.max(src).alias("mx")).collect()[0]
    mn, mx = stats["mn"], stats["mx"]
    if mn is None or mx is None or mn == mx:
        return df.withColumn(out, F.lit(50.0))   # no spread -> neutral
    scaled = (F.col(src) - F.lit(mn)) / (F.lit(mx) - F.lit(mn)) * 100
    return df.withColumn(out, (F.lit(100.0) - scaled) if invert else scaled)

# Assemble one row per company
base = (s("dim_company").select("company_id")
        .join(fin_base,"company_id","left")
        .join(mkt_base,"company_id","left")
        .join(gov_base,"company_id","left")
        .join(news_base,"company_id","left"))

# Fill nulls neutrally before scoring
num_cols = ["debt_to_equity","current_ratio","net_margin","revenue_growth",
            "annual_volatility","max_drawdown","beta",
            "company_age_years","accounts_overdue","director_churn",
            "news_volume","news_sentiment"]
base = base.fillna(0, subset=num_cols)

# ---- Financial pillar ---- (high debt = risk; high liquidity/margin/growth = safe -> invert)
base = norm(base,"debt_to_equity","s_de")
base = norm(base,"current_ratio","s_cr", invert=True)
base = norm(base,"net_margin","s_nm", invert=True)
base = norm(base,"revenue_growth","s_rg", invert=True)
base = base.withColumn("financial_score", (F.col("s_de")+F.col("s_cr")+F.col("s_nm")+F.col("s_rg"))/4)

# ---- Market pillar ---- (high vol/beta = risk; drawdown is negative so more negative = risk -> invert)
base = norm(base,"annual_volatility","s_vol")
base = norm(base,"max_drawdown","s_dd", invert=True)   # -0.6 worse than -0.1
base = norm(base,"beta","s_beta")
base = base.withColumn("market_score", (F.col("s_vol")+F.col("s_dd")+F.col("s_beta"))/3)

# ---- Governance pillar ---- (older = safer -> invert age; churn & overdue = risk)
base = norm(base,"company_age_years","s_age", invert=True)
base = norm(base,"director_churn","s_churn")
base = norm(base,"accounts_overdue","s_ovd")
base = base.withColumn("governance_score", (F.col("s_age")+F.col("s_churn")+F.col("s_ovd"))/3)

# ---- News pillar ---- (negative sentiment = risk -> invert sentiment; high volume = more attention/risk)
base = norm(base,"news_sentiment","s_sent", invert=True)
base = norm(base,"news_volume","s_vol_news")
base = base.withColumn("news_score", (F.col("s_sent")+F.col("s_vol_news"))/2)

# ---- Composite ----
base = base.withColumn("risk_score",
    0.35*F.col("financial_score") + 0.30*F.col("market_score")
  + 0.25*F.col("governance_score") + 0.10*F.col("news_score"))
base = base.withColumn("risk_band",
    F.when(F.col("risk_score")>=66,"High").when(F.col("risk_score")>=33,"Medium").otherwise("Low"))

display(base.select("company_id","financial_score","market_score","governance_score",
                    "news_score","risk_score","risk_band").orderBy(F.col("risk_score").desc()))

In [0]:



# =========================================================
# CELL 11 — Assemble fact_company_risk and MERGE
# =========================================================
score_date = spark.sql("SELECT current_date() AS d").collect()[0]["d"]
score_key = int(score_date.strftime("%Y%m%d"))

ind_map = (s("yf_info").select(F.col("ticker"), F.trim("sector").alias("sector_name"),
                               F.trim("industry").alias("industry_name"))
           .join(s("dim_company").select("company_id","ticker"),"ticker","inner")
           .join(spark.read.table(f"{CATALOG}.gold.dim_industry"),
                 ["sector_name","industry_name"],"left")
           .select("company_id","industry_id"))

fact = (base
    .join(ind_map,"company_id","left")
    .withColumn("officer_summary_id", F.col("company_id"))
    .withColumn("score_date_key", F.lit(score_key))
    .withColumn("score_date", F.lit(score_date))
    .select(
        "company_id","score_date_key","industry_id","officer_summary_id",
        "debt_to_equity","current_ratio","net_margin","revenue_growth","financial_score",
        "annual_volatility","max_drawdown","beta","market_score",
        "company_age_years","accounts_overdue","governance_score",
        "news_volume","news_sentiment","news_score",
        "risk_score","risk_band","score_date",
    )
)

scd_merge(fact, f"{CATALOG}.gold.fact_company_risk", ["company_id"])



In [0]:
# =========================================================
# CELL 12 — Validate final output
# =========================================================
result = (spark.read.table(f"{CATALOG}.gold.fact_company_risk")
          .join(spark.read.table(f"{CATALOG}.gold.dim_company").select("company_id","company_name"),
                "company_id")
          .select("company_name","financial_score","market_score","governance_score",
                  "news_score","risk_score","risk_band")
          .orderBy(F.col("risk_score").desc()))
display(result)

print("Companies scored:", spark.read.table(f"{CATALOG}.gold.fact_company_risk").count())